In [26]:
# ============================================== CONFIG & USER DEFINED FUNCTIONS ======================================================= #

from pyspark.sql import functions as F, Window
from pyspark.sql.types import *
from delta.tables import DeltaTable
from datetime import datetime
import uuid

# =========================================================
# CONFIG
# =========================================================
def get_config(environment="dev"):
    config = {
        "dev": {
            "raw_base_path": "Files/raw",
            "bronze_schema": "dbo",
            "silver_schema": "dbo",
            "gold_schema": "dbo"
        },
        "preprod": {
            "raw_base_path": "Files/raw",
            "bronze_schema": "dbo",
            "silver_schema": "dbo",
            "gold_schema": "dbo"
        },
        "prod": {
            "raw_base_path": "Files/raw",
            "bronze_schema": "dbo",
            "silver_schema": "dbo",
            "gold_schema": "dbo"
        }
    }
    return config[environment]

CONTROL_LOG_TABLE = "ETL.pl_control_config_log"

def generate_run_id():
    return str(uuid.uuid4())

# =========================================================
# STEP LOGGING
# =========================================================
def log_step_start(
    run_id,
    pipeline_name,
    pipeline_step_name,
    source_file_path,
    target_table_name,
    pipeline_start_time
):
    schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("pipeline_name", StringType(), True),
        StructField("pipeline_step_name", StringType(), True),
        StructField("source_file_path", StringType(), True),
        StructField("target_table_name", StringType(), True),
        StructField("step_status", StringType(), True),
        StructField("pipeline_status", StringType(), True),
        StructField("record_count", IntegerType(), True),
        StructField("error_message", StringType(), True),
        StructField("step_start_time", TimestampType(), True),
        StructField("step_end_time", TimestampType(), True),
        StructField("pipeline_start_time", TimestampType(), True),
        StructField("pipeline_end_time", TimestampType(), True),
        StructField("created_at", TimestampType(), True),
        StructField("updated_at", TimestampType(), True)
    ])

    now = datetime.now()

    row = [(
        run_id,
        pipeline_name,
        pipeline_step_name,
        source_file_path,
        target_table_name,
        "STARTED",
        "STARTED",
        0,
        None,
        now,
        None,
        pipeline_start_time,
        None,
        now,
        now
    )]

    spark.createDataFrame(row, schema) \
        .write.format("delta") \
        .mode("append") \
        .saveAsTable(CONTROL_LOG_TABLE)

def log_step_end(
    run_id,
    pipeline_name,
    pipeline_step_name,
    step_status,
    pipeline_status,
    record_count,
    error_message=None,
    pipeline_end_time=None
):
    error_message = (error_message or "").replace("'", " ")

    if pipeline_end_time is None:
        pipeline_end_sql = "NULL"
    else:
        pipeline_end_sql = f"TIMESTAMP('{pipeline_end_time}')"

    spark.sql(f"""
        UPDATE {CONTROL_LOG_TABLE}
        SET step_status = '{step_status}',
            pipeline_status = '{pipeline_status}',
            record_count = {record_count},
            error_message = '{error_message}',
            step_end_time = current_timestamp(),
            pipeline_end_time = {pipeline_end_sql},
            updated_at = current_timestamp()
        WHERE run_id = '{run_id}'
          AND pipeline_name = '{pipeline_name}'
          AND pipeline_step_name = '{pipeline_step_name}'
    """)

def finalize_pipeline_run(run_id, pipeline_name, final_status):
    spark.sql(f"""
        UPDATE {CONTROL_LOG_TABLE}
        SET pipeline_status = '{final_status}',
            pipeline_end_time = current_timestamp(),
            updated_at = current_timestamp()
        WHERE run_id = '{run_id}'
          AND pipeline_name = '{pipeline_name}'
    """)

# =========================================================
# RESTART / SKIP LOGIC
# =========================================================
def should_run_step(run_id, pipeline_name, pipeline_step_name, rerun_mode="FULL"):
    if rerun_mode.upper() == "FULL":
        return True

    df = spark.sql(f"""
        SELECT step_status
        FROM {CONTROL_LOG_TABLE}
        WHERE run_id = '{run_id}'
          AND pipeline_name = '{pipeline_name}'
          AND pipeline_step_name = '{pipeline_step_name}'
        ORDER BY updated_at DESC
        LIMIT 1
    """)

    rows = df.collect()

    if len(rows) == 0:
        return True

    latest_status = rows[0]["step_status"]

    if rerun_mode.upper() == "FAILED_ONLY" and latest_status in ("SUCCESS", "SUCCESS_WITH_REJECTS"):
        return False

    return True

# =========================================================
# DATE PARSER
# =========================================================
def parse_multi_format_timestamp(col_name):
    return F.coalesce(
        F.to_timestamp(F.col(col_name), "yyyy-MM-dd"),
        F.to_timestamp(F.col(col_name), "yyyy-MM-dd HH:mm:ss"),
        F.to_timestamp(F.col(col_name), "dd/MM/yyyy"),
        F.to_timestamp(F.col(col_name), "MM/dd/yyyy"),
        F.to_timestamp(F.col(col_name), "yyyy/MM/dd"),
        F.to_timestamp(F.col(col_name), "dd-MM-yyyy"),
        F.to_timestamp(F.col(col_name))
    )

# =========================================================
# MERGE UPSERT
# =========================================================
def merge_upsert(df, target_table_name, merge_condition):
    if spark.catalog.tableExists(target_table_name):
        delta_table = DeltaTable.forName(spark, target_table_name)
        (
            delta_table.alias("t")
            .merge(df.alias("s"), merge_condition)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        df.write.format("delta").mode("overwrite").saveAsTable(target_table_name)

# =========================================================
# SILVER HELPERS
# =========================================================
def first_existing(df, candidate_cols):
    for c in candidate_cols:
        if c in df.columns:
            return c
    return None

def col_or_null(df, candidate_cols, alias_name, cast_type=None, transform=None):
    chosen = first_existing(df, candidate_cols)

    if chosen is None:
        expr = F.lit(None)
    else:
        expr = F.col(chosen)

    if transform == "trim":
        expr = F.trim(expr)
    elif transform == "lower_trim":
        expr = F.lower(F.trim(expr))
    elif transform == "initcap_trim":
        expr = F.initcap(F.trim(expr))
    elif transform == "timestamp":
        if chosen is None:
            expr = F.lit(None).cast("timestamp")
        else:
            expr = parse_multi_format_timestamp(chosen)

    if cast_type:
        expr = expr.cast(cast_type)

    return expr.alias(alias_name)

def write_error_table(df_error, target_table, run_id):
    df_out = (
        df_error.withColumn("reject_ts", F.current_timestamp())
                .withColumn("pipeline_run_id", F.lit(run_id))
    )
    df_out.write.format("delta").mode("overwrite").saveAsTable(target_table)

# ============================================== CONFIG & USER DEFINED FUNCTIONS ======================================================= #

StatementMeta(, 4ae7088e-3de5-42cf-b853-452883a403c7, 55, Finished, Available, Finished, False)

In [27]:
from pyspark.sql import functions as F, Window
from datetime import datetime

# =========================================================
# PARAMETERS
# =========================================================
environment = globals().get("environment", "dev")

default_run_id = "45e6d27f-77cb-49a0-b1ae-7eecf73012bb"
run_id = globals().get("run_id", default_run_id)

rerun_mode = globals().get("rerun_mode", "FULL")
pipeline_name = "pl_retail_analytics_medallion"
pipeline_start_time = globals().get("pipeline_start_time", datetime.now())

if run_id is None:
    raise Exception("run_id must be passed from Bronze/Silver")

print(f"environment = {environment}")
print(f"run_id = {run_id}")
print(f"rerun_mode = {rerun_mode}")

StatementMeta(, 4ae7088e-3de5-42cf-b853-452883a403c7, 56, Finished, Available, Finished, False)

environment = dev
run_id = 45e6d27f-77cb-49a0-b1ae-7eecf73012bb
rerun_mode = FULL


In [28]:
# =========================================================
# 1. SILVER CUSTOMERS
# =========================================================
step_name = "silver_customers"
source_table = "dbo.bronze_customers"
target_table = "dbo.silver_customers"
reject_table = "dbo.silver_customers_error"

if should_run_step(run_id, pipeline_name, step_name, rerun_mode):
    try:
        log_step_start(run_id, pipeline_name, step_name, source_table, target_table, pipeline_start_time)

        df = spark.read.table(source_table)
        source_count = df.count()

        standardized = df.select(
            col_or_null(df, ["customer_id", "cust_id", "customer_code"], "customer_id", transform="trim"),
            col_or_null(df, ["customer_name", "name", "full_name"], "customer_name", transform="initcap_trim"),
            col_or_null(df, ["email", "email_id"], "email", transform="lower_trim"),
            col_or_null(df, ["country", "cust_country"], "country", transform="trim"),
            col_or_null(df, ["customer_created_date", "created_at", "signup_date"], "customer_created_ts", transform="timestamp"),
            F.col("ingestion_ts"),
            F.col("source_file_name"),
            F.col("pipeline_run_id")
        )

        error_df = standardized.filter(F.col("customer_id").isNull() | 
                                       F.col("email").isNull() | 
                                       F.col("country").isNull()) \
                            .withColumn("reject_reason",
                                        F.when(F.col("customer_id").isNull(), F.lit("customer_id is null"))
                                         .when(F.col("email").isNull(), F.lit("email is null"))
                                         .when(F.col("country").isNull(), F.lit("country is null"))
                                )

        valid_df = standardized.filter(F.col("customer_id").isNotNull() | 
                                       F.col("email").isNotNull() | 
                                       F.col("country").isNotNull())

        w = Window.partitionBy("customer_id").orderBy(F.col("ingestion_ts").desc())
        valid_df = valid_df.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

        valid_count = valid_df.count()
        bad_count = error_df.count()

        write_error_table(error_df, reject_table, run_id)
        merge_upsert(valid_df, target_table, "t.customer_id = s.customer_id")

        status = "SUCCESS_WITH_REJECTS" if bad_count > 0 else "SUCCESS"
        msg = f"source_count={source_count}; valid_count={valid_count}; bad_count={bad_count}; reject_table={reject_table}"

        log_step_end(run_id, pipeline_name, step_name, status, "IN_PROGRESS", valid_count, msg)
        print(f"{step_name} completed")

    except Exception as e:
        log_step_end(run_id, pipeline_name, step_name, "FAILED", "FAILED", 0, str(e), datetime.now())
        raise
else:
    print(f"{step_name} skipped")

# =========================================================
# 2. SILVER PRODUCTS
# =========================================================
step_name = "silver_products"
source_table = "dbo.bronze_products"
target_table = "dbo.silver_products"
reject_table = "dbo.silver_products_error"

if should_run_step(run_id, pipeline_name, step_name, rerun_mode):
    try:
        log_step_start(run_id, pipeline_name, step_name, source_table, target_table, pipeline_start_time)

        df = spark.read.table(source_table)
        source_count = df.count()

        standardized = df.select(
            col_or_null(df, ["product_id", "prod_id", "sku"], "product_id", transform="trim"),
            col_or_null(df, ["product_name", "name"], "product_name", transform="initcap_trim"),
            col_or_null(df, ["category", "product_category"], "category", transform="initcap_trim"),
            col_or_null(df, ["unit_price", "price"], "unit_price", cast_type="decimal(18,2)"),
            F.col("ingestion_ts"),
            F.col("source_file_name"),
            F.col("pipeline_run_id")
        )

        error_df = standardized.filter(F.col("product_id").isNull()) \
            .withColumn("reject_reason", F.lit("product_id is null"))

        valid_df = standardized.filter(F.col("product_id").isNotNull())

        w = Window.partitionBy("product_id").orderBy(F.col("ingestion_ts").desc())
        valid_df = valid_df.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

        valid_count = valid_df.count()
        bad_count = error_df.count()

        write_error_table(error_df, reject_table, run_id)
        merge_upsert(valid_df, target_table, "t.product_id = s.product_id")

        status = "SUCCESS_WITH_REJECTS" if bad_count > 0 else "SUCCESS"
        msg = f"source_count={source_count}; valid_count={valid_count}; bad_count={bad_count}; reject_table={reject_table}"

        log_step_end(run_id, pipeline_name, step_name, status, "IN_PROGRESS", valid_count, msg)
        print(f"{step_name} completed")

    except Exception as e:
        log_step_end(run_id, pipeline_name, step_name, "FAILED", "FAILED", 0, str(e), datetime.now())
        raise
else:
    print(f"{step_name} skipped")

# =========================================================
# 3. SILVER ORDERS
# =========================================================
step_name = "silver_orders"
source_table = "dbo.bronze_orders"
target_table = "dbo.silver_orders"
reject_table = "dbo.silver_orders_error"

if should_run_step(run_id, pipeline_name, step_name, rerun_mode):
    try:
        log_step_start(run_id, pipeline_name, step_name, source_table, target_table, pipeline_start_time)

        df = spark.read.table(source_table)
        source_count = df.count()

        standardized = df.select(
            col_or_null(df, ["order_id", "ord_id"], "order_id", transform="trim"),
            col_or_null(df, ["customer_id", "cust_id"], "customer_id", transform="trim"),
            col_or_null(df, ["order_date", "order_ts", "order_datetime"], "order_ts", transform="timestamp"),
            col_or_null(df, ["status", "order_status"], "order_status", transform="initcap_trim"),
            F.col("ingestion_ts"),
            F.col("source_file_name"),
            F.col("pipeline_run_id")
        )

        error_df = standardized.filter(
            F.col("order_id").isNull() |
            F.col("customer_id").isNull() |
            F.col("order_ts").isNull() |
            F.col("order_status").isNull()
        ).withColumn(
            "reject_reason",
            F.when(F.col("order_id").isNull(), F.lit("order_id is null"))
             .when(F.col("customer_id").isNull(), F.lit("customer_id is null"))
             .when(F.col("order_status").isNull(), F.lit("order_status is null"))
             .otherwise(F.lit("order_ts invalid/null"))
        )

        valid_df = standardized.filter(
            F.col("order_id").isNotNull() &
            F.col("customer_id").isNotNull() &
            F.col("order_ts").isNotNull() &
            F.col("order_status").isNotNull()
        )

        w = Window.partitionBy("order_id").orderBy(F.col("ingestion_ts").desc())
        valid_df = valid_df.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

        valid_count = valid_df.count()
        bad_count = error_df.count()

        write_error_table(error_df, reject_table, run_id)
        merge_upsert(valid_df, target_table, "t.order_id = s.order_id")

        status = "SUCCESS_WITH_REJECTS" if bad_count > 0 else "SUCCESS"
        msg = f"source_count={source_count}; valid_count={valid_count}; bad_count={bad_count}; reject_table={reject_table}"

        log_step_end(run_id, pipeline_name, step_name, status, "IN_PROGRESS", valid_count, msg)
        print(f"{step_name} completed")

    except Exception as e:
        log_step_end(run_id, pipeline_name, step_name, "FAILED", "FAILED", 0, str(e), datetime.now())
        raise
else:
    print(f"{step_name} skipped")

# =========================================================
# 4. SILVER ORDER ITEMS
# =========================================================
step_name = "silver_order_items"
source_table = "dbo.bronze_order_items"
target_table = "dbo.silver_order_items"
reject_table = "dbo.silver_order_items_error"

if should_run_step(run_id, pipeline_name, step_name, rerun_mode):
    try:
        log_step_start(run_id, pipeline_name, step_name, source_table, target_table, pipeline_start_time)

        df = spark.read.table(source_table)
        products = spark.read.table("dbo.silver_products").select("product_id").distinct()
        source_count = df.count()

        standardized = df.select(
            col_or_null(df, ["order_id"], "order_id", transform="trim"),
            col_or_null(df, ["product_id"], "product_id", transform="trim"),
            col_or_null(df, ["quantity", "qty"], "quantity", cast_type="int"),
            F.col("ingestion_ts"),
            F.col("source_file_name"),
            F.col("pipeline_run_id")
        )

        joined = standardized.join(
            products.withColumnRenamed("product_id", "valid_product_id"),
            standardized.product_id == F.col("valid_product_id"),
            "left"
        )

        error_df = joined.filter(
            F.col("order_id").isNull() |
            F.col("product_id").isNull() |
            F.col("quantity").isNull() |
            (F.col("quantity") <= 0) |
            F.col("valid_product_id").isNull()
        ).withColumn(
            "reject_reason",
            F.when(F.col("order_id").isNull(), F.lit("order_id is null"))
             .when(F.col("product_id").isNull(), F.lit("product_id is null"))
             .when(F.col("quantity").isNull(), F.lit("quantity is null"))
             .when(F.col("quantity") <= 0, F.lit("quantity <= 0"))
             .otherwise(F.lit("invalid product_id"))
        )

        valid_df = joined.filter(
            F.col("order_id").isNotNull() &
            F.col("product_id").isNotNull() &
            F.col("quantity").isNotNull() &
            (F.col("quantity") > 0) &
            F.col("valid_product_id").isNotNull()
        ).drop("valid_product_id")

        valid_df = valid_df.dropDuplicates(["order_id", "product_id"])

        valid_count = valid_df.count()
        bad_count = error_df.count()

        write_error_table(error_df, reject_table, run_id)
        merge_upsert(valid_df, target_table, "t.order_id = s.order_id AND t.product_id = s.product_id")

        status = "SUCCESS_WITH_REJECTS" if bad_count > 0 else "SUCCESS"
        msg = f"source_count={source_count}; valid_count={valid_count}; bad_count={bad_count}; reject_table={reject_table}"

        log_step_end(run_id, pipeline_name, step_name, status, "IN_PROGRESS", valid_count, msg)
        print(f"{step_name} completed")

    except Exception as e:
        log_step_end(run_id, pipeline_name, step_name, "FAILED", "FAILED", 0, str(e), datetime.now())
        raise
else:
    print(f"{step_name} skipped")

# =========================================================
# 5. SILVER RETURNS
# =========================================================
step_name = "silver_returns"
source_table = "dbo.bronze_returns"
target_table = "dbo.silver_returns"
reject_table = "dbo.silver_returns_error"

if should_run_step(run_id, pipeline_name, step_name, rerun_mode):
    try:
        log_step_start(run_id, pipeline_name, step_name, source_table, target_table, pipeline_start_time)

        df = spark.read.table(source_table)
        source_count = df.count()

        standardized = df.select(
            col_or_null(df, ["return_id"], "return_id", transform="trim"),
            col_or_null(df, ["order_id"], "order_id", transform="trim"),
            col_or_null(df, ["return_date", "return_ts"], "return_ts", transform="timestamp"),
            F.col("ingestion_ts"),
            F.col("source_file_name"),
            F.col("pipeline_run_id")
        )

        error_df = standardized.filter(
            F.col("return_id").isNull() |
            F.col("order_id").isNull() |
            F.col("return_ts").isNull() 
        ).withColumn(
            "reject_reason",
            F.when(F.col("return_id").isNull(), F.lit("return_id is null"))
             .when(F.col("order_id").isNull(), F.lit("order_id is null"))
             .when(F.col("return_ts").isNull(), F.lit("return_ts invalid/null"))
        )

        valid_df = standardized.filter(
            F.col("return_id").isNotNull() &
            F.col("order_id").isNotNull() &
            F.col("return_ts").isNotNull() 
        )

        valid_df = valid_df.dropDuplicates(["return_id"])

        valid_count = valid_df.count()
        bad_count = error_df.count()

        write_error_table(error_df, reject_table, run_id)
        merge_upsert(valid_df, target_table, "t.return_id = s.return_id")

        status = "SUCCESS_WITH_REJECTS" if bad_count > 0 else "SUCCESS"
        msg = f"source_count={source_count}; valid_count={valid_count}; bad_count={bad_count}; reject_table={reject_table}"

        log_step_end(run_id, pipeline_name, step_name, status, "IN_PROGRESS", valid_count, msg)
        print(f"{step_name} completed")

    except Exception as e:
        log_step_end(run_id, pipeline_name, step_name, "FAILED", "FAILED", 0, str(e), datetime.now())
        raise
else:
    print(f"{step_name} skipped")


print("Silver transformation completed successfully")
print(f"pipeline_run_id = {run_id}")

StatementMeta(, 4ae7088e-3de5-42cf-b853-452883a403c7, 57, Finished, Available, Finished, False)

silver_customers completed
silver_products completed
silver_orders completed
silver_order_items completed
silver_returns completed
Silver transformation completed successfully
pipeline_run_id = 45e6d27f-77cb-49a0-b1ae-7eecf73012bb


In [29]:
df = spark.sql("SELECT * FROM Retail_Analytics_Lakehouse.dbo.silver_customers LIMIT 1000")
display(df)

StatementMeta(, 4ae7088e-3de5-42cf-b853-452883a403c7, 58, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3980ca9f-71e6-4145-8233-e913de786518)